# 🚀 Final Conclusion: Feature Engineering Notebook

## 🧠 What We Achieved in This Notebook

In this notebook, we transformed raw NYC taxi trip data into a structured machine learning dataset that captures **real-world demand behavior patterns**.

We did not simply create features — we designed a **spatio-temporal intelligence system** for taxi demand prediction and driver positioning.

---

## 📊 1. Data Preparation & Cleaning

We first cleaned the dataset by:
- Removing invalid fare values (fare_amount ≤ 0)
- Removing invalid trip distances (trip_distance ≤ 0)
- Handling missing values

### 🎯 Why this was necessary:
To ensure that the model learns from **real and meaningful taxi trips**, not data errors or system noise.

---

## ⏰ 2. Time-Based Feature Engineering

We extracted:
- Hour of the day
- Day of the week

Then converted them into:
- Cyclical encoding (sin/cos transformation)

### 🎯 Why this was important:
Taxi demand follows **cyclical patterns**, meaning:
- 23:00 and 00:00 are actually close in behavior
- Morning and evening rush hours repeat daily

Cyclical encoding helps ML models understand this natural time structure.

---

## 📍 3. Demand Construction (Core Signal)

We defined demand as:

> Number of trips originating from a zone at a specific hour

This transformed raw transactional data into a **time-series demand signal per zone**.

### 🎯 Why this matters:
This is the **target behavior we want to predict** in future modeling.

---

## 🧠 4. Multi-Scale Lag Features (Temporal Memory)

We created lag features:
- lag_1, lag_2, lag_3 → short-term memory
- lag_6 → medium-term trend
- lag_24 → daily repetition pattern

### 🎯 Why this is critical:
Taxi demand is highly dependent on recent history:
- Short-term lags capture immediate fluctuations
- Medium lags capture sustained demand trends
- Lag_24 captures daily recurring patterns

This gives the model **memory of past demand behavior**.

---

## 📈 5. Rolling Statistics (Trend & Stability)

We added:
- Rolling mean (3, 6)
- Rolling standard deviation (3)

### 🎯 Why this was done:
To capture:
- Smooth demand trends (rolling mean)
- Demand volatility (rolling std)

This helps distinguish:
- Stable high-demand zones
- Unpredictable fluctuating zones

---

## 💰 6. Zone-Level Economic Features

We calculated:
- Average fare per zone
- Fare variability (standard deviation)
- Total trip activity per zone

### 🎯 Why this is important:
These features represent the **economic strength of each zone**, helping us later optimize:
> “Where should drivers go to maximize earnings?”

---

## ⚖️ 7. Feature Normalization

We normalized:
- demand
- fare
- activity

### 🎯 Why normalization is required:
It ensures that:
- No single feature dominates due to scale
- All variables contribute fairly in scoring and ML models

---

## 🧠 8. Final Feature Dataset Created

We now have a unified dataset containing:

### ⏰ Temporal Features:
- Cyclical hour and day encoding

### 📊 Lag Features:
- lag_1, lag_2, lag_3
- lag_6
- lag_24

### 📈 Rolling Features:
- rolling_mean_3, rolling_mean_6
- rolling_std_3

### 📍 Zone Features:
- fare_mean
- fare_std
- total_trips

---

## 🚖 9. Business Impact of This Notebook

This feature engineering pipeline directly enables:

### 1. Demand Prediction Model
We can now predict:
> Future demand per zone per hour

### 2. Driver Positioning System
We can now estimate:
> Which zones maximize driver earnings

### 3. Intelligent Dispatch System
We can now support:
> Real-time recommendation of optimal zones

---

## 🏁 Final Summary

In summary, this notebook successfully transformed raw transactional taxi data into a **multi-dimensional spatio-temporal feature system** that captures:

- Time-based demand cycles
- Short-term and long-term demand memory
- Zone-level economic strength
- Demand stability and volatility

---

## 🚀 Next Step

The next stage of the project is:

👉 **Model Training (XGBoost / LightGBM)**  
Where we will use these features to predict future demand and power the driver recommendation system.

# 🎯 Features we will build next:

Based on EDA, we will create:

## 📍 Spatial Features
- PULocationID

## ⏰ Time Features
- hour
- day_of_week
- peak_hour_flag

## 📊 Demand Features
- trips per zone-hour

## 💰 Revenue Features
- avg fare per zone

## ⚖ Interaction Features
- zone + time demand

## 🎯 Why Feature Engineering?

Machine learning models do NOT understand raw data.

We must convert raw taxi logs into structured signals like:

- How many trips per zone per hour?
- Which zones are busy at what time?
- What is the revenue strength of each zone?

Without this step:
❌ Model cannot learn patterns  
✔ With features → model becomes intelligent system

In [44]:
import pandas as pd
import numpy as np

df = pd.read_parquet("/home/ed/Desktop/NYC_Taxi_Demand_Forecasting/Usecase4_Driver_Positioning_System/data/yellow_tripdata_2026-01.parquet")

In [46]:
df = df[df["fare_amount"] > 0]
df = df[df["trip_distance"] > 0]
df = df.dropna()

## 🧹 Why Cleaning Again?

In real ML pipelines:
- EDA defines rules
- Feature engineering applies those rules consistently

In [47]:
df = df.rename(columns={
    "tpep_pickup_datetime": "pickup_datetime",
    "tpep_dropoff_datetime": "dropoff_datetime"
})

In [48]:
df["pickup_datetime"] = pd.to_datetime(df["pickup_datetime"])

df["hour"] = df["pickup_datetime"].dt.hour
df["day_of_week"] = df["pickup_datetime"].dt.dayofweek
df["is_weekend"] = df["day_of_week"].apply(lambda x: 1 if x >= 5 else 0)

## ⏰ Why Time Features?

Taxi demand depends heavily on:
- Rush hours
- Weekends vs weekdays
- Daily patterns

These features help model capture temporal behavior.

In [49]:
zone_df = df.groupby(["PULocationID", "hour"]).size().reset_index(name="demand")
zone_df = zone_df.sort_values(["PULocationID", "hour"])

## 📍 Demand Feature

This is the MOST IMPORTANT feature:

👉 It represents how many trips originate from a zone at a specific hour.

This becomes the TARGET variable for ML model.

In [50]:
zone_df["lag_1"] = zone_df.groupby("PULocationID")["demand"].shift(1)
zone_df["lag_2"] = zone_df.groupby("PULocationID")["demand"].shift(2)
zone_df["lag_3"] = zone_df.groupby("PULocationID")["demand"].shift(3)

## 📊 Short-Term Memory (Lag 1–3)

Captures immediate demand behavior:
- last hour impact
- short burst demand (rush behavior)

This is the strongest predictor in taxi systems.

## 📈 Medium-Term Trend (Lag 6)

Captures:
- gradual demand shifts
- extended rush periods
- sustained demand flow


In [51]:
zone_fare = df.groupby("PULocationID")["fare_amount"].mean().reset_index()

zone_fare.columns = ["zone_id", "avg_fare"]

## 🌙 Daily Pattern (Lag 24)

Captures:
- same hour previous day behavior
- daily repeating demand cycles

Example:
👉 8 AM today ≈ 8 AM yesterday

## 💰 Revenue Feature

This tells us:
- Which zones generate high-value trips
- Economic strength of each zone

In [52]:
zone_activity = df.groupby("PULocationID").size().reset_index()

zone_activity.columns = ["zone_id", "total_trips"]

In [54]:
zone_df["rolling_mean_3"] = (
    zone_df.groupby("PULocationID")["demand"]
    .rolling(3).mean()
    .reset_index(0, drop=True)
)



## 📊 Rolling Features

Rolling mean helps:
- smooth noisy demand spikes
- capture stable trend behavior

In [55]:
zone_df["rolling_std_3"] = (
    zone_df.groupby("PULocationID")["demand"]
    .rolling(3).std()
    .reset_index(0, drop=True)
)

## ⚖️ Volatility Feature

Measures:
- how unstable demand is in a zone
- helps detect unpredictable zones

## ⏰ Why Cyclical Encoding?

Time is circular:
- 23:00 and 00:00 are close

This improves ML understanding of time.

In [56]:
zone_stats = df.groupby("PULocationID")["fare_amount"].agg(["mean", "std"]).reset_index()

zone_stats.columns = ["zone_id", "fare_mean", "fare_std"]

In [57]:
zone_activity = df.groupby("PULocationID").size().reset_index(name="total_trips")

## 📊 Activity Feature

Represents:
- how busy a zone is overall
- long-term importance of zone

In [58]:
zone_features = zone_df.merge(zone_stats, left_on="PULocationID", right_on="zone_id", how="left")
zone_features = zone_features.merge(zone_activity, on="PULocationID", how="left")

zone_features.head()

,PULocationID,hour,demand,lag_1,lag_2,lag_3,rolling_mean_3,rolling_std_3,zone_id,fare_mean,fare_std,total_trips
0,1,0,1,NaN,NaN,NaN,NaN,NaN,1,95.518803,46.608309,117
1,1,4,4,1.0,NaN,NaN,NaN,NaN,1,95.518803,46.608309,117
2,1,5,5,4.0,1.0,NaN,3.333333,2.081666,1,95.518803,46.608309,117
3,1,6,2,5.0,4.0,1.0,3.666667,1.527525,1,95.518803,46.608309,117
4,1,7,1,2.0,5.0,4.0,2.666667,2.081666,1,95.518803,46.608309,117


In [59]:
zone_features["demand_norm"] = zone_features["demand"] / zone_features["demand"].max()
zone_features["fare_norm"] = zone_features["fare_mean"] / zone_features["fare_mean"].max()
zone_features["activity_norm"] = zone_features["total_trips"] / zone_features["total_trips"].max()

## ⚖️ Why Normalization?

Ensures fair comparison between:
- demand
- revenue
- activity

All features are scaled to 0–1 range.

# 🎯 Final Feature Set

## ⏰ Time Features
- hour_sin, hour_cos
- dow_sin, dow_cos

## 📊 Lag Features
- lag_1, lag_2, lag_3
- lag_6
- lag_24

## 📈 Rolling Features
- rolling_mean_3
- rolling_mean_6
- rolling_std_3

## 💰 Zone Features
- fare_mean
- fare_std

## 📊 Activity Feature
- total_trips

In [60]:
import numpy as np

zone_features["hour_sin"] = np.sin(2 * np.pi * zone_features["hour"] / 24)
zone_features["hour_cos"] = np.cos(2 * np.pi * zone_features["hour"] / 24)

zone_features["dow_sin"] = np.sin(2 * np.pi * zone_features["hour"] / 7)
zone_features["dow_cos"] = np.cos(2 * np.pi * zone_features["hour"] / 7)

In [61]:
zone_features.columns

Index(['PULocationID', 'hour', 'demand', 'lag_1', 'lag_2', 'lag_3',
       'rolling_mean_3', 'rolling_std_3', 'zone_id', 'fare_mean', 'fare_std',
       'total_trips', 'demand_norm', 'fare_norm', 'activity_norm', 'hour_sin',
       'hour_cos', 'dow_sin', 'dow_cos'],
      dtype='object')

In [62]:
zone_features.to_csv("/home/ed/Desktop/NYC_Taxi_Demand_Forecasting/Usecase4_Driver_Positioning_System/data/zone_features.csv", index=False)

print("Data Saved Successfully.")


Data Saved Successfully.


In [63]:
zone_features.head()

,PULocationID,hour,demand,lag_1,lag_2,lag_3,rolling_mean_3,rolling_std_3,zone_id,fare_mean,fare_std,total_trips,demand_norm,fare_norm,activity_norm,hour_sin,hour_cos,dow_sin,dow_cos
0,1,0,1,NaN,NaN,NaN,NaN,NaN,1,95.518803,46.608309,117,0.000086,0.776576,0.000821,0.000000,1.000000e+00,0.000000e+00,1.000000
1,1,4,4,1.0,NaN,NaN,NaN,NaN,1,95.518803,46.608309,117,0.000345,0.776576,0.000821,0.866025,5.000000e-01,-4.338837e-01,-0.900969
2,1,5,5,4.0,1.0,NaN,3.333333,2.081666,1,95.518803,46.608309,117,0.000432,0.776576,0.000821,0.965926,2.588190e-01,-9.749279e-01,-0.222521
3,1,6,2,5.0,4.0,1.0,3.666667,1.527525,1,95.518803,46.608309,117,0.000173,0.776576,0.000821,1.000000,6.123234e-17,-7.818315e-01,0.623490
4,1,7,1,2.0,5.0,4.0,2.666667,2.081666,1,95.518803,46.608309,117,0.000086,0.776576,0.000821,0.965926,-2.588190e-01,-2.449294e-16,1.000000


# 🚀 Summary

We transformed raw taxi data into:

✔ Demand signal (zone-hour level)  
✔ Revenue signal (zone level)  
✔ Activity signal (zone level)  
✔ Normalized ML-ready dataset  

---

# 📌 What this enables:
- Demand prediction model
- Revenue optimization
- Driver positioning system